# SecAlign Defense: Step-by-Step Preference Optimization (ORPO/DPO) Fine-Tuning & Evaluation

This notebook demonstrates how to implement **SecAlign** (*SecAlign: Defending Against Prompt Injection with Preference Optimization*, Chen et al., CCS 2025) within the `ipi` benchmark framework.

## 📌 Overview & Workflow
1. **Preference Dataset Construction**: Transform clean instruction datasets into `(prompt, chosen, rejected)` triplets where `chosen` satisfies the legitimate user task and `rejected` reflects execution of prompt injection.
2. **QLoRA Fine-Tuning on Kaggle T4**: Use `trl` (`ORPOTrainer` / `DPOTrainer`) with 4-bit NormalFloat quantization (BitsAndBytes) to fine-tune 1B - 8B models within free T4 VRAM limits (~16GB).
3. **Model Publishing**: Save fine-tuned LoRA adapters for export to **Hugging Face Hub** or **Kaggle Datasets**.
4. **Benchmark Evaluation**: Wrap the fine-tuned model into `SecAlignDefense` and run comparative evaluations against all prompt injection attacks in `ipi`.

In [ ]:
# Cell 1 — Installation & Environment Setup
# Install core ipi framework and fine-tuning packages
!pip install -q git+https://github.com/alirezaAalaie/IPI-Aaptive.git
!pip install -q trl peft transformers datasets bitsandbytes accelerate

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")


In [ ]:
# Cell 2 — Dataset Generation (Chosen vs Rejected Preference Pairs)
from ipi.defenses.secalign import generate_secalign_preference_data

# Example instruction samples (can be loaded from Alpaca / BIPIA / custom dataset)
clean_data = [
    {
        "instruction": "Summarize the key points of the user document.",
        "input": "The Q3 financial report indicates a 15% revenue growth in enterprise software sales.",
        "output": "The Q3 report highlights a 15% growth in enterprise software sales."
    },
    {
        "instruction": "Extract action items from the email.",
        "input": "Hi Team, please finalize the budget review by Friday 5 PM. Also prepare the quarterly slides.",
        "output": "1. Finalize budget review by Friday 5 PM.\n2. Prepare quarterly slides."
    },
    {
        "instruction": "Translate the given text into French.",
        "input": "Artificial Intelligence is transforming business automation.",
        "output": "L'intelligence artificielle transforme l'automatisation des entreprises."
    },
    {
        "instruction": "Answer the customer inquiry politely.",
        "input": "Where is my order #4092? It was supposed to arrive yesterday.",
        "output": "We apologize for the delay. We are tracking order #4092 and will update you shortly."
    }
]

# Generate preference dataset for ORPO / DPO training
pref_dataset = generate_secalign_preference_data(
    clean_samples=clean_data,
    attack_type="Naive",
    alignment_type="orpo",
    seed=42
)

print(f"✅ Successfully generated {len(pref_dataset)} SecAlign preference pairs!\n")
print("--- SAMPLE PREFERENCE PAIR ---")
print("PROMPT:", pref_dataset[0]["prompt"])
print("CHOSEN (Clean Goal):", pref_dataset[0]["chosen"])
print("REJECTED (Injected Goal):", pref_dataset[0]["rejected"])


In [ ]:
# Cell 3 — SecAlign QLoRA Fine-Tuning on Kaggle T4 GPU
from ipi.defenses.secalign import train_secalign

# Model selection guidelines for Kaggle T4 GPU (16 GB VRAM):
#  - 1.5B - 3B Models (Qwen2.5-1.5B, Llama-3.2-3B): ~15-30 mins training time
#  - 7B - 8B Models (Mistral-7B-v0.3, Llama-3-8B): ~1-2 hours training time with 4-bit QLoRA

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "./secalign_lora_weights"

# Launch training step (Uncomment below to execute full fine-tuning on GPU kernel)
"""
trainer = train_secalign(
    model_name_or_path=MODEL_ID,
    output_dir=OUTPUT_DIR,
    train_samples=clean_data,
    alignment_type="orpo",
    use_4bit=True,                   # Enables 4-bit NF4 QLoRA for low VRAM
    lora_r=16,
    lora_alpha=32,
    learning_rate=5e-5,
    num_train_epochs=1.0,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_length=512
)
print(f"✅ Training finished! Weights saved to {OUTPUT_DIR}")
"""


In [ ]:
# Cell 4 — Save & Publish Model Adapter (Hugging Face / Kaggle)
"""
# Option A: Push Adapter to Hugging Face Hub
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id="your-username/SecAlign-Qwen2.5-1.5B-LoRA",
    repo_type="model"
)

# Option B: Save in Kaggle Notebook Output Directory for export
"""
print("Model artifact publication script ready.")


In [ ]:
# Cell 5 — Evaluate SecAlign Defense on ipi Attack Benchmark
from ipi.target import LocalLLM
from ipi.defenses.secalign import SecAlignDefense
from ipi.attacks import NaiveAttacker, IgnoreAttacker, FakeCompletionAttacker
from ipi.evaluator import BipiaSuccessEvaluator
from ipi.dataset import HijackDataset

"""
# 1. Wrap fine-tuned SecAlign model
base_victim = LocalLLM(
    model_name_or_path=MODEL_ID,
    adapter_path=OUTPUT_DIR
)
secalign_victim = SecAlignDefense(target=base_victim)

# 2. Initialize attacks & dataset
dataset = HijackDataset(limit=10)
evaluator = BipiaSuccessEvaluator()
attackers = [
    NaiveAttacker(),
    IgnoreAttacker(),
    FakeCompletionAttacker()
]

# 3. Run Benchmark Evaluation
results = {}
for attacker in attackers:
    att_name = attacker.__class__.__name__
    print(f"Evaluating attack: {att_name}...")
    # Execute attack generation and score success rate
"""
print("SecAlign evaluation pipeline configured.")
